In [ ]:
import numpy as np,tensorflow as tf,matplotlib.pyplot as plt
np.random.seed(17);tf.random.set_seed(17)
# Character-level corpus; repeat a small public-domain-style sample to form sequences
corpus=("it was the best of times it was the worst of times deep learning models learn patterns from data. " * 250)
chars=sorted(set(corpus));char_to_id={c:i for i,c in enumerate(chars)};id_to_char={i:c for c,i in char_to_id.items()}
encoded=np.array([char_to_id[c] for c in corpus],dtype=np.int32);seq_len=40
starts=np.arange(0,len(encoded)-seq_len-1,3);X=np.array([encoded[i:i+seq_len] for i in starts]);y=np.array([encoded[i+seq_len] for i in starts])
X=tf.keras.utils.to_categorical(X,len(chars));y=tf.keras.utils.to_categorical(y,len(chars))
def train_with_clipping(name,clipnorm=None,clipvalue=None):
 model=tf.keras.Sequential([tf.keras.layers.Input((seq_len,len(chars))),tf.keras.layers.SimpleRNN(64),tf.keras.layers.Dense(len(chars),activation="softmax")])
 opt=tf.keras.optimizers.Adam(learning_rate=.03,clipnorm=clipnorm,clipvalue=clipvalue)
 model.compile(optimizer=opt,loss="categorical_crossentropy")
 ds=tf.data.Dataset.from_tensor_slices((X,y)).shuffle(2000,seed=17).batch(64)
 norms=[];losses=[]
 for step,(xb,yb) in enumerate(ds.take(100)):
  with tf.GradientTape() as tape:
   pred=model(xb,training=True);loss=tf.reduce_mean(tf.keras.losses.categorical_crossentropy(yb,pred))
  grads=tape.gradient(loss,model.trainable_variables);norms.append(float(tf.linalg.global_norm(grads)));losses.append(float(loss))
  opt.apply_gradients(zip(grads,model.trainable_variables))
  if not np.isfinite(loss):print(name,"non-finite loss at step",step);break
 print(name,"max global gradient norm:",round(float(np.nanmax(norms)),4),"final loss:",round(float(losses[-1]),4),"non-finite steps:",int(np.sum(~np.isfinite(losses))))
 return np.array(norms),np.array(losses)
settings=[("No clipping",None,None),("clipnorm=1.0",1.0,None),("clipnorm=0.5",.5,None),("clipvalue=1.0",None,1.0),("clipvalue=0.5",None,.5)]
curves={}
for name,cn,cv in settings:curves[name]=train_with_clipping(name,cn,cv)
fig,ax=plt.subplots(1,2,figsize=(13,4))
for name,(gn,ls) in curves.items():ax[0].plot(gn,label=name);ax[1].plot(ls,label=name)
ax[0].set_title("Global gradient norm by training step");ax[1].set_title("Loss by training step")
for a in ax:a.set_xlabel("Training step");a.legend();a.grid()
plt.tight_layout();plt.show()
print("Compare maximum gradient norms above; clipping thresholds constrain updates and reduce large spikes. Select a threshold using stable loss and validation behaviour.")
